# 01. Минимальный первичный EDA для Entity Resolution

Цель ноутбука — быстро понять базовую структуру train-датасета перед построением витрин и моделей:

- сколько строк, профилей и `entity_id`;
- есть ли проблемы с ключами `profile_id` и `entity_id`;
- сколько в данных дублей: в сущностях, профилях и pair-представлении;
- насколько заполнены identity-поля;
- какие сырые feature-колонки надо разбирать дальше.

Детальный parsing `realtime_features`, `fs_features`, `non_processing_features`, blocking-анализ и model-level оценка вынесены в следующие ноутбуки.

In [ ]:
from pathlib import Path
from math import comb

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

RAW_DATA_DIR = Path('../data/raw')
REPORT_DIR = Path('../reports/01_eda')
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DATA_DIR / 'split_label_train_V3.snappy.parquet'
DATA_PATH


## 1. Загрузка данных

Одна строка исходного parquet — это событие/наблюдение профиля. Дальше в проекте мы агрегируем данные до уровня `profile_id`, но здесь сначала смотрим на исходную гранулярность.

In [ ]:
df = pd.read_parquet(DATA_PATH, engine='pyarrow')
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce') if 'created_at' in df.columns else pd.NaT

print('shape:', df.shape)
print('columns:', df.shape[1])
display(df.head(3))


## 2. Схема и пропуски

Проверяем, что ожидаемые колонки есть в датасете, и строим короткий column-level отчет: тип, число пропусков, доля пропусков и число уникальных значений.

In [ ]:
expected_columns = [
    'profile_id', 'entity_id', 'created_at',
    'first_name', 'last_name', 'email', 'phone', 'birthday', 'sex',
    'realtime_features', 'fs_features', 'non_processing_features',
]

schema_report = pd.DataFrame({
    'column': expected_columns,
    'exists': [col in df.columns for col in expected_columns],
})

def safe_nunique(series):
    try:
        return int(series.nunique(dropna=True))
    except TypeError:
        return np.nan


column_overview = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[col].dtype) for col in df.columns],
    'non_null_rows': [int(df[col].notna().sum()) for col in df.columns],
    'null_rows': [int(df[col].isna().sum()) for col in df.columns],
    'null_share': [float(df[col].isna().mean()) for col in df.columns],
    'nunique_scalar': [safe_nunique(df[col]) for col in df.columns],
}).sort_values('null_share', ascending=False)

display(schema_report)
display(column_overview)


## 3. Гранулярность: строки, профили, entity

Здесь фиксируем базовые размеры датасета и проверяем важный sanity-check: один `profile_id` не должен ссылаться на несколько разных `entity_id`. Если такие строки есть, это риск для дальнейшей разметки пар.

In [ ]:
base_counts = pd.DataFrame([
    {'metric': 'rows', 'value': len(df)},
    {'metric': 'profiles', 'value': df['profile_id'].nunique(dropna=True)},
    {'metric': 'entities', 'value': df['entity_id'].nunique(dropna=True)},
    {'metric': 'min_created_at', 'value': df['created_at'].min()},
    {'metric': 'max_created_at', 'value': df['created_at'].max()},
])

profile_event_counts = df.groupby('profile_id', dropna=False).size().rename('event_count')
profile_event_summary = profile_event_counts.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame('event_count')

profile_entity_nunique = df.groupby('profile_id', dropna=False)['entity_id'].nunique(dropna=True).rename('entity_id_nunique')
profile_entity_consistency = pd.DataFrame([
    {'metric': 'profiles_with_0_entity_id', 'value': int(profile_entity_nunique.eq(0).sum())},
    {'metric': 'profiles_with_1_entity_id', 'value': int(profile_entity_nunique.eq(1).sum())},
    {'metric': 'profiles_with_gt1_entity_id', 'value': int(profile_entity_nunique.gt(1).sum())},
])

display(base_counts)
display(profile_event_summary)
display(profile_entity_consistency)


## 4. Сколько дублей в датасете

Дубли можно считать на нескольких уровнях, и эти проценты нельзя смешивать:

- `duplicate_entities` — сколько `entity_id` имеют больше одного профиля;
- `profiles_in_duplicate_entities` — сколько профилей лежат в таких duplicated entity;
- `extra_duplicate_profiles_over_first` — сколько профилей являются «дополнительными» сверх первого профиля в entity;
- `positive_pairs` — сколько существует пар профилей с одинаковым `entity_id`.

Для pair-модели особенно важен `positive_pairs`, но его доля среди всех возможных пар почти всегда очень мала.

In [ ]:
profile_core = (
    df.groupby('profile_id', dropna=False)
    .agg(entity_id=('entity_id', 'first'))
    .reset_index()
)

entity_sizes = profile_core.groupby('entity_id', dropna=False)['profile_id'].nunique().rename('entity_profile_count')

total_profiles = int(profile_core['profile_id'].nunique(dropna=True))
total_entities = int(entity_sizes.shape[0])
singleton_entities = int(entity_sizes.eq(1).sum())
duplicate_entities = int(entity_sizes.gt(1).sum())
profiles_in_singletons = int(entity_sizes[entity_sizes.eq(1)].sum())
profiles_in_duplicate_entities = int(entity_sizes[entity_sizes.gt(1)].sum())
extra_duplicate_profiles = int((entity_sizes[entity_sizes.gt(1)] - 1).sum())
positive_pairs = int(sum(comb(int(n), 2) for n in entity_sizes if n > 1))
total_possible_pairs = total_profiles * (total_profiles - 1) // 2

summary_rows = [
    {'metric': 'total_profiles', 'count': total_profiles, 'share': 1.0},
    {'metric': 'total_entities', 'count': total_entities, 'share': 1.0},
    {'metric': 'singleton_entities', 'count': singleton_entities, 'share': singleton_entities / total_entities},
    {'metric': 'duplicate_entities', 'count': duplicate_entities, 'share': duplicate_entities / total_entities},
    {'metric': 'profiles_in_singleton_entities', 'count': profiles_in_singletons, 'share': profiles_in_singletons / total_profiles},
    {'metric': 'profiles_in_duplicate_entities', 'count': profiles_in_duplicate_entities, 'share': profiles_in_duplicate_entities / total_profiles},
    {'metric': 'extra_duplicate_profiles_over_first', 'count': extra_duplicate_profiles, 'share': extra_duplicate_profiles / total_profiles},
    {'metric': 'positive_pairs', 'count': positive_pairs, 'share': positive_pairs / total_possible_pairs},
    {'metric': 'total_possible_profile_pairs', 'count': total_possible_pairs, 'share': 1.0},
]
duplicate_summary = pd.DataFrame(summary_rows)

entity_size_distribution = (
    entity_sizes.value_counts()
    .sort_index()
    .rename_axis('entity_profile_count')
    .reset_index(name='n_entities')
)
entity_size_distribution['entity_share'] = entity_size_distribution['n_entities'] / total_entities

display(duplicate_summary)
display(entity_size_distribution)


## 5. Заполненность identity-полей

На этом этапе не строим признаки и не нормализуем подробно. Только смотрим, какие identity-поля заполнены и где внутри одного `profile_id` встречается несколько разных значений.

In [ ]:
identity_cols = [col for col in ['first_name', 'last_name', 'email', 'phone', 'birthday', 'sex'] if col in df.columns]

identity_profile_report = []
for col in identity_cols:
    by_profile = df.groupby('profile_id', dropna=False)[col].nunique(dropna=True)
    identity_profile_report.append({
        'column': col,
        'filled_event_share': float(df[col].notna().mean()),
        'profiles_with_value': int(by_profile.gt(0).sum()),
        'profile_coverage': float(by_profile.gt(0).mean()),
        'profiles_with_multiple_values': int(by_profile.gt(1).sum()),
        'multi_value_profile_share': float(by_profile.gt(1).mean()),
    })

identity_profile_report = pd.DataFrame(identity_profile_report).sort_values('profile_coverage', ascending=False)
display(identity_profile_report)


## 6. Сырые feature-колонки

`realtime_features`, `fs_features`, `non_processing_features` оставляем для следующих ноутбуков. Здесь достаточно понять, насколько они заполнены, сколько у них сырых уникальных значений, и увидеть пару примеров формата.

In [ ]:
feature_columns = [col for col in ['realtime_features', 'fs_features', 'non_processing_features'] if col in df.columns]
feature_column_report = []
for col in feature_columns:
    non_null = df[col].notna()
    as_text = df.loc[non_null, col].astype(str)
    feature_column_report.append({
        'column': col,
        'filled_rows': int(non_null.sum()),
        'filled_share': float(non_null.mean()),
        'nunique_as_text': int(as_text.nunique(dropna=True)),
        'example_1': as_text.iloc[0] if len(as_text) else None,
        'example_2': as_text.iloc[1] if len(as_text) > 1 else None,
    })

feature_column_report = pd.DataFrame(feature_column_report)
display(feature_column_report)


## 7. Сохраняем короткие EDA-отчеты

Сохраняем только компактные CSV-отчеты, которые полезны как входной контроль перед следующими шагами.

In [ ]:
report_files = {
    'base_counts.csv': base_counts,
    'column_overview.csv': column_overview,
    'duplicate_summary.csv': duplicate_summary,
    'entity_size_distribution.csv': entity_size_distribution,
    'identity_profile_report.csv': identity_profile_report,
    'feature_column_report.csv': feature_column_report,
}

for name, report in report_files.items():
    report.to_csv(REPORT_DIR / name, index=False)

print('Saved reports:')
for name in report_files:
    print('-', name)


## 8. Что важно вынести из EDA

Минимальный вывод из этого ноутбука:

- датасет event-level, поэтому для ER нужна profile-level агрегация;
- `entity_id` используется как label/ground truth, а не production-признак;
- доля positive pairs среди всех возможных пар крайне мала, поэтому полный перебор пар невозможен;
- identity-поля полезны, но их заполненность и стабильность надо проверять на profile-level;
- сложные feature-колонки надо разбирать отдельно в следующих ноутбуках.